# **Multi-Asset Portfolio Performance, Risk & Attribution**

**FINM3422 — Assessment 2 - Group 8**

# **1.0 Introduction**

This report presents a performance, risk, and attribution analysis of a large Australian superannuation fund investing across five asset-class sleeves: Australian Equities (AUS EQ), International Equities (INTL EQ), Bonds / Fixed Income, Real Estate (RE), and Private Equity / Venture Capital (PE/VC). Each sleeve is managed by an external manager and evaluated against a designated benchmark index.

The analysis covers the 10-year period from January 2016 to December 2025 using monthly return data, assessed against a composite benchmark constructed from Strategic Asset Allocation (SAA) weights. Performance is evaluated at both sleeve and total fund level, with APRA-inspired supervisory checks applied to assess long-run return adequacy, volatility, drawdown resilience, and stress scenario behaviour. Active return is further decomposed using the Brinson attribution framework to identify the  contribution of allocation and selection decisions across all five sleeves.

# **2.0 Data Overview**

The dataset consists of monthly return series for five asset-class sleeves, covering both manager returns and corresponding benchmark indices over the period January 2016 to December 2025 (120 observations). Benchmarks reflect industry-standard proxies: the S&P/ASX 200 for AUS EQ, MSCI World ex-Australia for INTL EQ, a Bloomberg AusBond proxy for Bonds, an A-REIT proxy for Real Estate, and a stylised synthetic benchmark for PE/VC. A risk-free rate proxy is included for Sharpe ratio calculations.

All data is loaded via a centralised pipeline that enforces a DatetimeIndex, validates monthly frequency, confirms full manager-benchmark date alignment, and checks for missing values prior to analysis.

Two sets of weights are used throughout — the Tactical Asset Allocation (TAA), reflecting the fund's current active positioning, and the Strategic Asset Allocation (SAA), representing the long-term policy benchmark. Both are held constant across the analysis period:

| Asset Class            | TAA Weight | SAA Weight |
|------------------------|------------|------------|
| Australian Equities    | 35%        | 40%        |
| International Equities | 35%        | 30%        |
| Bonds                  | 15%        | 20%        |
| Real Estate            | 5%         | 5%         |
| PE/VC                  | 10%        | 5%         |

### 2.1 Environment Info, Imports and Confit

In [1]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

sys.path.append("../src")

from data_loader import load_all_data, SLEEVE_LABELS, TAA_WEIGHTS, SAA_WEIGHTS
from performance import (all_sleeves_summary, sleeve_summary, annualised_return, annualised_volatility, max_drawdown, apra_checks, equity_crash, bond_yield_spike, display_summary_tables, fund_vs_benchmark)
from attribution import brinson_attribution
from charts import (plot_sleeve_wealth_index, plot_fund_vs_benchmark, plot_sharpe_bar, plot_ir_bar, plot_apra_drawdown_threshold, plot_attribution)

print(sys.version)
print("pandas:", pd.__version__, "numpy:", np.__version__)
pd.set_option("display.float_format", "{:.4f}".format)

ImportError: cannot import name 'brinson_attribution' from 'attribution' (/Users/ameliaperry/Desktop/FINM3422-Group-8-Task-2/notebook/../src/attribution.py)

### 2.2 Parameters and Paths

In [ ]:
# Base path to data folder
BASE_path = "/Users/ameliaperry/Desktop/BAFE/FINM3422/python case/FINM3422-Group-8-Task-2-1/data/"

# TAA and SAA weights loaded from data_loader
taa_weights = pd.Series(TAA_WEIGHTS)
saa_weights = pd.Series(SAA_WEIGHTS)

### 2.3 Data Load and Sanity Check

In [ ]:
# Load and Validate Data
data = load_all_data(BASE_path)

df_managers   = data["managers"]
df_benchmarks = data["benchmarks"]
df_rf         = data["rf"]
# Sanity Checks
print("Any nulls in benchmarks?", df_benchmarks.isna().sum().sum())
print("Any nulls in managers?", df_managers.isna().sum().sum())
print("Dates aligned?", df_benchmarks.index.equals(df_managers.index))
print("Monotonic dates?", df_benchmarks.index.is_monotonic_increasing)
#
display(df_managers.head())
display(df_benchmarks.head())
display(df_rf.head()) 

# **3.0 Performance and Risk Analysis**

### 3.1 Methodology

Performance metrics are calculated using monthly returns and annualised 
geometrically to capture the compound growth rate. Together, these metrics 
provide a comprehensive view of both absolute and relative performance, 
covering return adequacy, risk efficiency, and downside resilience.

**Annualised Return** — geometric growth rate of monthly returns:
$$R_{\text{ann}} = \left(\prod_{t=1}^{n}(1 + r_t)\right)^{12/n} - 1$$
- $r_t$: monthly return in month $t$, $n$: number of monthly observations

**Annualised Volatility** — monthly standard deviation scaled annually:
$$\sigma_{\text{ann}} = \sigma_{\text{monthly}} \times \sqrt{12}$$

**Sharpe Ratio** — excess return above the risk-free rate per unit of total risk:
$$\text{Sharpe} = \frac{R_{\text{ann}} - R_{f,\text{ann}}}{\sigma_{\text{ann}}}$$
- $R_{f,\text{ann}}$: annualised risk-free rate

**Tracking Error** — annualised volatility of monthly active returns:
$$TE = \sigma(r_P - r_B) \times \sqrt{12}$$
- $r_P - r_B$: monthly active return (manager minus benchmark)

**Information Ratio** — active return per unit of active risk:
$$IR = \frac{R_P - R_B}{TE}$$

**Maximum Drawdown** — largest peak-to-trough decline in cumulative wealth:
$$MDD = \min_t \left(\frac{W_t - \max(W)}{\max(W)}\right)$$
- $W_t$: cumulative wealth at time $t$


### 3.2 Results Summary

In [ ]:
summary = all_sleeves_summary(df_managers, df_benchmarks, df_rf)
display_summary_tables(summary)

Accross the five sleeves, manager returns were consistently above the relative benchmarks, suggesting overall positive active management over the full 10-year sample period. However, the level and consistency of outperformance varied significantly between sleeves. 

**Australian Equities** showed modest outperformance (6.91% vs 5.88%), with a low Sharpe index of 0.27, reflective of high volatility compared to return. Maximum drawdown of -24.63% is consistent with the significant Australian equity market fluctuations over this time. 

**International Equities** performed the strongest on both an absolute and risk-adjusted returns basis. The manager delivered 15.57% annualsied against a 13.94% benchmark while maintaining the lowest volatility of the equity sleeves at 11.96%. The Sharpe ratio of 1.05 is the only sleeve to achieve above 1.0, indicating genuine risk-adjusted perfromance. The low tracking error of 2.57% alongside an IR of 0.64 suggests the manager was disciplined and actve by consistently added value without taking large bets.

**Bonds** produced near-zero risk-adjusted returns, with the Sharpe Ratio of -0.00 reflecting that the return of 3.02% was only minorly above the risk-free rate over this period. This was to be expected, as the firs half of the sample period's low-yield environment compressed bond yields. The 2022 rate hiking cycle generated significant mark-to-market losses, reflectd in the -11.35 maximum drawdown. Despite this, the bonds manager produced the highest IR of 1.21 across all sleeves, outfperforming the benchmark by 1.13% with a tracking error of only 0.94%. Even though bonds achieved the lowest return, the manager for this sleeve was the most effective by adding substantial value relative to the benchmark despite the challenging rate environment.

**Real Estate** produced the second highest manager return at 10.84% value versus the benchmark of 8.03%, an outperformance of 2.81%. However, this also had the highest volatility of 18.72% and largest manager drawdown of -28.69% across akk sleeves. The tracking error almost three times larger than international equities of 7.10% indicates the manager took significant active risk to generate this return, also reflected in the relatively low IR of 0.39. The benchmarks lower drawdown of -37.41% suggests the manager navigated downside risk during periods of market stress, which is the primary source of outperformance. 

**PE/VC** delivered 10.24% annualised return with the lowest volatility of the sleeves at 10.86% and the second highest Sharpe ratio of 0.66. This smooth return profile is typical of private market assets, where valuations are reported infrequently and do not always accurately capture short-term market movements. The low tracking error of 2.08% and IR of 0.41 suggest steady, consistent outperformance rather than large active tilts.


### 3.3 Visualisations

In [ ]:
plot_sleeve_wealth_index(df_managers, df_benchmarks, SLEEVE_LABELS)
plot_sharpe_bar(summary.T)
plot_ir_bar(summary.T)

The cumulative return charts reinforce the summary table findings and provide further context on the timing and consistency of outperformance.

**Equity sleeves (AUS EQ & INTL EQ)** both show a sharp drawdown in early 2020 consistent with the COVID-19 shock, followed by a quicker recovery by the manager than the benchmark. INTL EQ shows the largest divergence, with the manager accelerating well above the benchmark from 2023 onwards to reach $4.20 versus $3.70 by 2026, the widest  terminal gap of all five sleeves and the primary driver of its 15.57% annualised return.

**Real Estate** experienced the most severe benchmark drawdown in 2020, falling back to approximately $1.00 while the manager held above $1.50. This justifies the large difference in maximum drawdown (-28.69% vs -37.41%). Both recovered strongly after 2021, with the manager sustaining a wide and growing gap through to 2026.

**Bonds and PE/VC** both show smooth, steady growth with the manager's returns consistently above the benchmark throughout the entire period and no visible severe drawdowns. The Bonds gap widens significantly after 2022, consistent with the manager navigating the rate hiking cycle more effectively than the benchmark. PE/VC follows the benchmark for most of the period before a divergence of approximately $0.25 emerges from 2024 onwards.

**Sharpe and IR charts** confirm that International Equities and PE/VC are the strongest risk-adjusted performers on both measures, while AUS EQ is low on both. The Bonds IR standing well above all other sleeves despite its near-zero Sharpe highlights that absolute and relative performance metrics are highly relevant on the manager's skill.